# Kiểm tra cấu trúc bộ Le2i Fall Detection

**Mục tiêu:** trước khi chuẩn hóa Le2i cùng UR Fall, xác nhận trên dữ liệu thật:
1. **Cấu trúc thư mục:** có những scene nào (Home, Coffee room, Office, Lecture room), thư mục `Videos` / `Annotation(s)_files` đặt ở đâu, tên có khác nhau giữa các scene không.
2. **Ghép video ↔ annotation:** video nào thiếu annotation, annotation nào không có video, số video có bị **trùng giữa các scene** không.
3. **Thông số video:** fps, kích thước, số frame (đọc thuộc tính và **đếm thật bằng giải mã**).
4. **Định dạng annotation:** 2 dòng đầu (frame bắt đầu/kết thúc ngã), các dòng bbox, **thứ tự cột bbox** (`x1,y1,x2,y2` hay `y1,x1,y2,x2`), frame đánh số từ 0 hay 1.
5. **Bằng mắt:** ảnh ghép có vẽ bbox lên frame, cho từng scene (1 video Fall + 1 video ADL).
6. **Phạm vi đã chốt: chỉ dùng Home + Coffee room.** Office và Lecture room bị loại (annotation của hai scene này bị mất). Notebook vẫn quét chúng để **kiểm chứng** lý do (có thiếu annotation thật không), nhưng không tính vào phạm vi dùng.

Notebook chỉ **đọc** dữ liệu, ghi kết quả vào `OUT_DIR`. **Cách dùng:** Add Input dataset Le2i → Run All → đọc ô *Tổng kết* ở cuối và xem ảnh ghép.

In [ ]:
import os, re, json
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
os.environ.setdefault("OPENCV_FFMPEG_LOGLEVEL", "-8")   # tắt cảnh báo "mp3float Header missing" của track audio trong .avi
os.environ.setdefault("OPENCV_LOG_LEVEL", "ERROR")
import cv2

# ================= CẤU HÌNH =================
DATA_ROOT = os.environ.get("LE2I_ROOT")   # None = tự tìm. Muốn điền tay thì viết thẳng DATA_ROOT = "/kaggle/input/.../ten_dataset" (KHÔNG bọc trong os.environ.get)
COUNT_BY_DECODING = False   # True: giải mã hết video để đếm frame thật. Lần kiểm tra đầu trên Le2i đã cho kết quả khớp 190/190 video nên tắt để nhẹ máy
FRAME_BASE = None           # frame trong annotation bắt đầu từ 0 hay 1? None = tự phát hiện
BBOX_LAYOUT = None          # "xyxy" hoặc "yxyx"; None = tự phát hiện
MARGIN = 15                 # số frame lấy thêm trước/sau khoảng ngã khi vẽ ảnh ghép
ROWS_PER_FIG = 4
EXCLUDED_GROUPS = {"office", "lecture_room"}   # ĐÃ CHỐT LOẠI: annotation của 2 scene này bị mất
EXPECTED = dict(videos_full=191, fps=25, size=(320, 240), groups={"home", "coffee_room"})
OUT_DIR = Path("/kaggle/working/check_le2i") if Path("/kaggle").exists() else Path("./check_le2i")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OpenCV", cv2.__version__, "| OUT_DIR =", OUT_DIR.resolve())

## 1. Tìm thư mục gốc và xem cây thư mục

In [ ]:
VIDEO_EXT = {".avi", ".mp4", ".mov", ".mkv"}

def find_root():
    if DATA_ROOT:
        return Path(DATA_ROOT)
    base = Path("/kaggle/input")
    vdirs = []
    if base.exists():
        for dp, dn, _ in os.walk(base):
            if Path(dp).name.lower() in ("videos", "video"):
                vdirs.append(Path(dp)); dn[:] = []
    if not vdirs:
        raise FileNotFoundError("Không thấy thư mục 'Videos' nào trong /kaggle/input. Hãy điền DATA_ROOT thủ công.")
    root = Path(os.path.commonpath([str(p) for p in vdirs]))
    return root.parent if root.name.lower() in ("videos", "video") else root

ROOT = find_root()
print("Thư mục gốc Le2i:", ROOT, "\n")

def print_tree(p, depth=0, max_depth=4):
    kids = sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
    dirs, files = [k for k in kids if k.is_dir()], [k for k in kids if k.is_file()]
    pad = "   " * depth
    for d in dirs:
        n = sum(1 for f in d.rglob("*") if f.is_file())
        print(f"{pad}[+] {d.name}/   ({n} file)")
        if depth + 1 < max_depth:
            print_tree(d, depth + 1, max_depth)
    if files:
        print(f"{pad}    {len(files)} file: " + ", ".join(f.name for f in files[:3]) + (" ..." if len(files) > 3 else ""))

print_tree(ROOT)
for rd in sorted(ROOT.glob("*.txt"))[:3]:
    print(f"\n===== Nội dung {rd.name} (tối đa 40 dòng) =====")
    print("\n".join(rd.read_text(encoding="utf-8", errors="ignore").splitlines()[:40]))

## 2. Kiểm kê video và annotation
Tên scene được chuẩn hóa về 4 nhóm (`home`, `coffee_room`, `office`, `lecture_room`) + phần `_01/_02`. Video ghép với annotation theo **(scene, phần, số thứ tự trong tên file)**, vì `video (31).avi` có thể trùng số giữa các scene.

In [ ]:
SCENE_RE = re.compile(r"^(coffee|home|office|lecture)[ _\-]*(?:room)?[ _\-]*(\d+)?$", re.I)
GROUP = {"coffee": "coffee_room", "home": "home", "office": "office", "lecture": "lecture_room"}

def scene_of(path):
    for part in reversed(path.relative_to(ROOT).parts[:-1]):      # tổ tiên gần nhất khớp tên scene
        m = SCENE_RE.match(part.strip())
        if m:
            return GROUP[m.group(1).lower()], (m.group(2) or ""), part
    return "unknown", "", ""

def file_no(stem):
    m = re.search(r"\((\d+)\)", stem)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", stem)
    return int(nums[-1]) if nums else None

def in_annot_dir(p):
    return any(re.search(r"annot", part, re.I) for part in p.relative_to(ROOT).parts[:-1])

vid_files = sorted(p for p in ROOT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXT)
txt_files = sorted(ROOT.rglob("*.txt"))
ann_files = [p for p in txt_files if in_annot_dir(p)]
other_txt = [p for p in txt_files if not in_annot_dir(p)]

def mk(p):
    g, part, sf = scene_of(p)
    n = file_no(p.stem)
    return dict(path=str(p), name=p.name, group=g, part=part, scene_folder=sf, no=n,
                key=f"{g}|{part}|{n}", pattern=re.sub(r"\d+", "N", p.name))

vids = pd.DataFrame([mk(p) for p in vid_files])
anns = pd.DataFrame([mk(p) for p in ann_files])

print(f"Video: {len(vids)} | annotation (.txt trong thư mục *annot*): {len(anns)} | .txt khác: {len(other_txt)}")
print("Đuôi video      :", Counter(p.suffix.lower() for p in vid_files))
print("Mẫu tên video   :", Counter(vids.pattern).most_common(5))
print("Mẫu tên annot   :", Counter(anns.pattern).most_common(5) if len(anns) else "(không có)")
print("Tên thư mục chứa annotation:", Counter(p.parent.name for p in ann_files))
print("Tên thư mục chứa video      :", Counter(p.parent.name for p in vid_files))
print("Thư mục KHÔNG khớp tên scene nào:", sorted(set(vids[vids.group == "unknown"].path.map(lambda s: str(Path(s).parent))))[:5] or "không có")
print("Video không đọc được số thứ tự:", vids[vids.no.isna()].name.tolist() or "không có")

dup_v = vids[vids.duplicated("key", keep=False)].sort_values("key")
dup_a = anns[anns.duplicated("key", keep=False)].sort_values("key") if len(anns) else anns
print("\nVideo trùng khóa (scene+số) :", "không có" if dup_v.empty else f"{len(dup_v)} file\n{dup_v[['key','path']].head(10).to_string()}")
print("Annotation trùng khóa       :", "không có" if dup_a.empty else f"{len(dup_a)} file")

pair = vids.merge(anns[["key", "path"]].drop_duplicates("key").rename(columns={"path": "ann_path"}), on="key", how="left")
pair["has_annot"] = pair.ann_path.notna()
orphans = anns[~anns.key.isin(vids.key)] if len(anns) else anns
print(f"\nVideo CÓ annotation: {int(pair.has_annot.sum())}/{len(pair)} | annotation KHÔNG có video: {len(orphans)}")
print(pair.groupby(["group", "part"]).agg(video=("path", "size"), co_annot=("has_annot", "sum")).to_string())
if len(orphans):
    print("\nAnnotation mồ côi (ví dụ):", orphans.path.head(5).tolist())

## 3. Thông số video + đọc annotation
`n_frames_prop` là số frame do container báo; `n_frames_dec` là số frame **giải mã thật** (đáng tin hơn). Annotation: 2 dòng đầu = frame bắt đầu/kết thúc ngã; các dòng sau = bbox từng frame (số cột sẽ được in ra).

In [ ]:
def probe(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return dict(opened=False)
    fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fc = int(cap.get(cv2.CAP_PROP_FOURCC))
    codec = "".join(chr((fc >> 8 * i) & 255) for i in range(4)).strip("\x00 ")
    nd = None
    if COUNT_BY_DECODING:
        nd = 0
        while cap.grab():
            nd += 1
    cap.release()
    return dict(opened=True, fps=round(fps, 3), n_frames_prop=n, n_frames_dec=nd, width=w, height=h, codec=codec)

NUM = re.compile(r"[-+]?\d*\.?\d+")
def parse_ann(path):
    lines = [l.strip() for l in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines() if l.strip()]
    out = dict(raw_head=lines[:4], n_lines=len(lines), start=None, end=None, header_ok=False, body=None, ncols=None)
    if len(lines) >= 2:
        a, b = NUM.findall(lines[0]), NUM.findall(lines[1])
        if len(a) == 1 and len(b) == 1:
            out.update(start=int(float(a[0])), end=int(float(b[0])), header_ok=True)
    rows = [list(map(float, NUM.findall(l))) for l in (lines[2:] if out["header_ok"] else lines)]
    rows = [r for r in rows if len(r) >= 5]
    if rows:
        ncols = Counter(len(r) for r in rows).most_common(1)[0][0]
        out["ncols"] = ncols
        out["body"] = np.array([r for r in rows if len(r) == ncols])
    return out

recs, ANN = [], {}
for _, v in pair.iterrows():
    r = v.to_dict()
    r.update(probe(v.path))
    if v.has_annot:
        a = parse_ann(v.ann_path)
        ANN[v.key] = a
        r.update(start=a["start"], end=a["end"], header_ok=a["header_ok"], n_lines=a["n_lines"],
                 ncols=a["ncols"], n_bbox=0 if a["body"] is None else len(a["body"]))
    recs.append(r)
inv = pd.DataFrame(recs)
inv["n_frames"] = inv.n_frames_dec.fillna(inv.n_frames_prop)

print("Không mở được:", int((~inv.opened).sum()), "video")
print("fps          :", inv.fps.value_counts().to_dict())
print("Kích thước   :", inv.groupby(["width", "height"]).size().to_dict())
print("fps theo scene:", {k: v for k, v in inv.groupby(["group", "part"]).fps.apply(lambda s: sorted(set(s.dropna()))).items()})
_wh = inv.assign(wh=inv.width.astype("Int64").astype(str) + "x" + inv.height.astype("Int64").astype(str))
print("Kích thước theo scene (số video):")
print(_wh.groupby(["group", "part"]).wh.value_counts().to_string())
print("Codec        :", inv.codec.value_counts().to_dict())
if COUNT_BY_DECODING:
    mis = inv[(inv.n_frames_prop != inv.n_frames_dec) & inv.opened]
    print(f"Số frame báo ≠ số frame giải mã thật: {len(mis)} video", "" if mis.empty else "(ví dụ: " + str(mis[['name','n_frames_prop','n_frames_dec']].head(3).values.tolist()) + ")")
print("Số cột của dòng bbox:", inv.ncols.value_counts(dropna=False).to_dict())
print("\nMẪU 4 dòng đầu của vài file annotation (để đọc bằng mắt):")
for k in list(ANN)[:3] + [k for k in ANN if ANN[k]["start"] not in (0, None)][:3]:
    print(f"  {k}:", ANN[k]["raw_head"])
print("\nAnnotation có 2 dòng đầu không phải 2 số đơn:", [k for k, a in ANN.items() if not a["header_ok"]][:10] or "không có")

## 4. Phân loại fall / ADL, kiểm tra khoảng ngã
Quy ước kiểm tra: `start>0` và `end>0` ⇒ video **có té ngã**; `start=end=0` ⇒ **không té ngã (ADL)**; trường hợp khác bị đánh dấu `ODD`.

In [ ]:
def label_of(r):
    if not r.has_annot: return "no_annot"
    if not r.header_ok: return "odd"
    if r.start > 0 and r.end > 0: return "fall" if r.end >= r.start else "odd"
    if r.start == 0 and r.end == 0: return "adl"
    return "odd"
inv["label"] = inv.apply(label_of, axis=1)
inv["excluded"] = inv.group.isin(EXCLUDED_GROUPS)
print("Phân loại:", inv.label.value_counts().to_dict())
print("Các cặp (start,end) đặc biệt ở video không phải fall:",
      inv[inv.label.isin(["adl", "odd"])][["start", "end"]].drop_duplicates().head(8).values.tolist())
for _, r in inv[inv.label == "odd"].iterrows():
    print("  video ODD", r.key, ":", ANN[r.key]["raw_head"], "| số dòng =", ANN[r.key]["n_lines"])

inv["end_over_n"] = (inv.label == "fall") & (inv.end > inv.n_frames)
fall_len = (inv.end - inv.start)[inv.label == "fall"]
print(f"\nKhoảng ngã dài (frame): min={fall_len.min()}, median={fall_len.median()}, max={fall_len.max()} ", "(quy đổi sang giây: chia cho 25)")
print("Khoảng ngã vượt quá số frame của video:", int(inv.end_over_n.sum()), "video")

## 5. Thứ tự cột bbox và cách đánh số frame
- **Thứ tự cột:** lấy 4 cột cuối của mỗi dòng bbox. Người **đứng** thì cao hơn rộng, nên nhìn các frame **trước khi ngã**: cách đọc nào cho `cao/rộng > 1` là cách đúng. Có thêm phép kiểm tra bbox nằm trong khung hình.
- **Đánh số frame:** so khoảng frame trong annotation với số frame video: `1..N` (từ 1) hay `0..N-1` (từ 0).

In [ ]:
def to_xyxy(b, layout):
    a, bb, c, d = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    return np.stack([a, bb, c, d], 1) if layout == "xyxy" else np.stack([bb, a, d, c], 1)

votes, valid = Counter(), {"xyxy": [0, 0], "yxyx": [0, 0]}
asp = {"xyxy": [], "yxyx": []}
base_kinds = []
zero_total = box_total = 0
by_wh = {}
for _, r in inv[inv.label.isin(["fall", "adl"]) & ~inv.excluded].iterrows():
    a = ANN.get(r.key)
    if a is None or a["body"] is None or not r.opened:
        continue
    body = a["body"]; fr = body[:, 0]; bx = body[:, -4:]
    zero_total += int((bx == 0).all(1).sum()); box_total += len(bx)
    lo, hi, n = int(fr.min()), int(fr.max()), int(r.n_frames)
    base_kinds.append("1" if (lo == 1 and hi in (n, n - 1, n - 2)) or (lo == 1 and hi <= n) else ("0" if lo == 0 else "?"))
    for lay in ("xyxy", "yxyx"):
        x1, y1, x2, y2 = to_xyxy(bx, lay).T
        ok = (x2 > x1) & (y2 > y1) & (x1 >= 0) & (y1 >= 0) & (x2 <= r.width + 1) & (y2 <= r.height + 1)
        valid[lay][0] += int(ok.sum()); valid[lay][1] += len(ok)
        if lay == "xyxy":
            _k = f"{int(r.width)}x{int(r.height)}"; by_wh.setdefault(_k, [0, 0]); by_wh[_k][0] += int(ok.sum()); by_wh[_k][1] += len(ok)
        if r.label == "fall":
            pre = fr < r.start - 3
            if pre.any():
                w, h = (x2 - x1)[pre], (y2 - y1)[pre]
                asp[lay].append(float(np.median(h[w > 0] / w[w > 0])) if (w > 0).any() else np.nan)
for lay in ("xyxy", "yxyx"):
    v = np.array(asp[lay], dtype=float)
    votes[lay] = int((v > 1).sum())
    print(f"Layout {lay}: bbox hợp lệ {valid[lay][0]}/{valid[lay][1]} | trước khi ngã cao/rộng>1 ở {votes[lay]}/{len(v)} video fall (trung vị {np.nanmedian(v):.2f})")

print(f"Dòng bbox toàn 0 (frame không có người): {zero_total}/{box_total} ({100 * zero_total / max(1, box_total):.1f}%)")
print("Tỉ lệ bbox nằm trọn trong khung hình (xyxy) theo kích thước video:", {k: f"{v[0]}/{v[1]} ({100 * v[0] / max(1, v[1]):.1f}%)" for k, v in by_wh.items()})
if BBOX_LAYOUT is None:
    BBOX_LAYOUT = "xyxy" if votes["xyxy"] >= votes["yxyx"] else "yxyx"
    amb = abs(votes["xyxy"] - votes["yxyx"]) < max(3, 0.2 * max(1, len(asp["xyxy"])))
    print(f"\n=> Thứ tự cột bbox tự phát hiện: {BBOX_LAYOUT}", "  ⚠ KHÔNG chắc chắn, hãy xem ảnh ghép bên dưới" if amb else "")
if FRAME_BASE is None:
    c = Counter(base_kinds)
    FRAME_BASE = 1 if c["1"] >= c["0"] else 0
    print(f"=> Annotation đánh số frame từ {FRAME_BASE}  (phân bố: {dict(c)})")

## 6. Bảng tổng hợp theo scene
Đối chiếu với kế hoạch (≈191 video, ≈143 fall, 4 bối cảnh, 25 fps, 320×240).

In [ ]:
inv["video_id"] = inv.apply(lambda r: f"le2i_{r.group}{r.part}_{int(r.no):03d}" if pd.notna(r.no) else None, axis=1)
inv["issues"] = ""
inv.loc[~inv.opened, "issues"] += "[KHONG_MO_DUOC] "
inv.loc[~inv.has_annot, "issues"] += "[THIEU_ANNOT] "
inv.loc[inv.label == "odd", "issues"] += "[KHOANG_NGA_LA] "
inv.loc[inv.end_over_n, "issues"] += "[END_VUOT_SO_FRAME] "
inv.loc[(inv.label == "fall") & (inv.n_bbox.fillna(0) == 0), "issues"] += "[FALL_KHONG_CO_BBOX] "
inv.loc[inv.duplicated("video_id", keep=False), "issues"] += "[TRUNG_VIDEO_ID] "
inv.loc[inv.group == "unknown", "issues"] += "[KHONG_NHAN_RA_SCENE] "
inv["usable_event"] = inv.opened & inv.label.isin(["fall", "adl"]) & ~inv.end_over_n & ~inv.excluded

inv["wh"] = inv.apply(lambda r: f"{int(r.width)}x{int(r.height)}" if pd.notna(r.width) else None, axis=1)
import pickle   # LƯU TRẠNG THÁI: nếu kernel chết ở ô sau, chỉ cần chạy lại ô ảnh ghép / tổng kết, không phải chạy lại từ đầu
pickle.dump(dict(inv=inv, ANN=ANN, BBOX_LAYOUT=BBOX_LAYOUT, FRAME_BASE=FRAME_BASE, orphans=orphans, ROOT=ROOT,
                 EXPECTED=EXPECTED, EXCLUDED_GROUPS=EXCLUDED_GROUPS, MARGIN=MARGIN, ROWS_PER_FIG=ROWS_PER_FIG,
                 COUNT_BY_DECODING=COUNT_BY_DECODING), open(OUT_DIR / "state.pkl", "wb"))
print("Đã lưu trạng thái ->", OUT_DIR / "state.pkl")
tab = inv.groupby(["group", "part"]).agg(
    video=("path", "size"), co_annot=("has_annot", "sum"), fall=("label", lambda s: int((s == "fall").sum())),
    adl=("label", lambda s: int((s == "adl").sum())), thieu_annot=("has_annot", lambda s: int((~s).sum())),
    dung_duoc_su_kien=("usable_event", "sum"), fps=("fps", lambda s: sorted(set(s.dropna()))),
    kich_thuoc=("wh", lambda s: sorted(set(s.dropna()))), da_loai=("excluded", "max")).reset_index()
use = tab[~tab.da_loai]
tab.loc[len(tab)] = ["TỔNG (phạm vi dùng)", "", use.video.sum(), use.co_annot.sum(), use.fall.sum(), use.adl.sum(),
                     use.thieu_annot.sum(), use.dung_duoc_su_kien.sum(), "", "", False]
tab

## 7. Xem bằng mắt: bbox có đúng người không?
Mỗi scene một video **Fall** (5 khung: trước ngã → bắt đầu → giữa → kết thúc → sau) và một video **ADL** (5 khung rải đều). Khung xanh = bbox từ annotation theo thứ tự cột đã phát hiện. **Cần thấy bbox bao đúng người.** Nếu bbox lệch hoặc lật trục, đổi `BBOX_LAYOUT` / `FRAME_BASE` ở ô cấu hình rồi chạy lại. **Nếu kernel bị restart ở mục 7 hoặc 8**, chỉ cần chạy lại chính ô đó: trạng thái được nạp từ `state.pkl` (đã lưu ở cuối mục 6).

In [ ]:
if "inv" not in globals():      # kernel đã restart: nạp lại trạng thái đã lưu ở mục 6
    import os, re, json, pickle
    from pathlib import Path
    from collections import Counter
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    os.environ.setdefault("OPENCV_FFMPEG_LOGLEVEL", "-8")
    import cv2
    OUT_DIR = Path("/kaggle/working/check_le2i") if Path("/kaggle").exists() else Path("./check_le2i")
    if not (OUT_DIR / "state.pkl").exists():
        raise RuntimeError("Chưa có state.pkl (kernel chết trước mục 6). Hãy Run All lại từ đầu.")
    globals().update(pickle.load(open(OUT_DIR / "state.pkl", "rb")))
    print("Kernel đã restart: đã nạp lại trạng thái từ", OUT_DIR / "state.pkl")

def to_xyxy(b, layout):
    a, bb, c, d = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    return np.stack([a, bb, c, d], 1) if layout == "xyxy" else np.stack([bb, a, d, c], 1)

import subprocess, io, shutil, sys
from PIL import Image

def _find_ffmpeg():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None
FFMPEG = _find_ffmpeg()
print("ffmpeg:", FFMPEG or "(không có: dùng OpenCV nhưng chạy trong tiến trình riêng)")

_CV_CODE = ("import sys, cv2\ncap = cv2.VideoCapture(sys.argv[1])\n"
            "for _ in range(int(sys.argv[2])):\n    if not cap.grab():\n        sys.exit(2)\n"
            "ok, fr = cap.read()\nif not ok:\n    sys.exit(3)\n"
            "ok, buf = cv2.imencode('.png', fr)\nsys.stdout.buffer.write(buf.tobytes())\n")

def read_frame(path, idx):
    """Đọc frame idx (0-based) trong TIẾN TRÌNH RIÊNG (ffmpeg, bỏ audio). Nếu bộ giải mã lỗi thì chỉ tiến trình con chết, kernel vẫn sống."""
    idx = max(int(idx), 0)
    if FFMPEG:
        cmd = [FFMPEG, "-v", "error", "-an", "-i", str(path), "-vf", f"select=eq(n\\,{idx})",
               "-frames:v", "1", "-f", "image2pipe", "-vcodec", "png", "-"]
    else:
        cmd = [sys.executable, "-c", _CV_CODE, str(path), str(idx)]
    try:
        r = subprocess.run(cmd, capture_output=True, timeout=120)
    except subprocess.TimeoutExpired:
        print("  ⚠ quá thời gian:", Path(path).name, "frame", idx); return None
    if r.returncode != 0 or not r.stdout:
        print(f"  ⚠ đọc frame lỗi (mã {r.returncode}):", Path(path).name, "frame", idx, r.stderr[-200:].decode(errors="ignore"))
        return None
    return np.array(Image.open(io.BytesIO(r.stdout)).convert("RGB"))

def box_at(key, ann_frame):
    a = ANN.get(key)
    if a is None or a["body"] is None:
        return None
    m = a["body"][:, 0] == ann_frame
    if not m.any():
        return None
    return to_xyxy(a["body"][m][:1, -4:], BBOX_LAYOUT)[0]

def montage_items():
    items = []
    for (g, p), sub in inv[inv.usable_event].groupby(["group", "part"]):
        f = sub[(sub.label == "fall") & (sub.n_bbox > 0)]
        a = sub[sub.label == "adl"]
        if len(f): items.append(("fall", f.iloc[0]))
        if len(a): items.append(("adl", a.iloc[0]))
    seen = {r.video_id for _, r in items}
    small = inv[inv.usable_event & (inv.wh == "320x180") & (inv.label == "fall") & (inv.n_bbox > 0) & ~inv.video_id.isin(seen)]
    for _, r in small.head(2).iterrows():      # video 320x180: kiểm tra bbox có lệch trục y không
        items.append(("fall", r))
    for _, r in inv[(inv.label == "odd") & ~inv.excluded].iterrows():   # video không có 2 dòng đầu: xem có té ngã không
        items.append(("odd", r))
    return items

def frames_for(kind, r):
    n = int(r.n_frames); lo, hi = FRAME_BASE, n - 1 + FRAME_BASE
    if kind == "fall":
        s, e = int(r.start), int(r.end)
        fs = [s - MARGIN, s, (s + e) // 2, e, e + MARGIN]
    else:
        fs = list(np.linspace(lo, hi, 5).astype(int))
    return [int(np.clip(f, lo, hi)) for f in fs]

items = montage_items()
print(f"{len(items)} hàng ảnh (mỗi scene 1 fall + 1 adl). Bố cục bbox = {BBOX_LAYOUT}, frame bắt đầu từ {FRAME_BASE}.")
for k in range(0, len(items), ROWS_PER_FIG):
    chunk = items[k:k + ROWS_PER_FIG]
    fig, axes = plt.subplots(len(chunk), 5, figsize=(15, 3.0 * len(chunk)), squeeze=False)
    for ri, (kind, r) in enumerate(chunk):
        for ci, f in enumerate(frames_for(kind, r)):
            ax = axes[ri][ci]; ax.axis("off")
            img = read_frame(r.path, f - FRAME_BASE)
            if img is not None:
                ax.imshow(img)
                b = box_at(r.key, f)
                if b is not None:
                    ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="lime", lw=2))
            ax.set_title(f"{r.video_id} {r.wh} [{kind}] f={f}", fontsize=8)
    plt.tight_layout(); fig.savefig(OUT_DIR / f"le2i_montage_{k // ROWS_PER_FIG + 1:02d}.png", dpi=110); plt.show()

## 8. Tổng kết và xuất file
Xuất `le2i_inventory.csv` (mỗi video một dòng: `video_id` đề xuất, scene, annotation, fps, số frame, khoảng ngã, cờ lỗi) và `le2i_summary.json`.

In [ ]:
if "inv" not in globals():      # kernel đã restart: nạp lại trạng thái đã lưu ở mục 6
    import os, re, json, pickle
    from pathlib import Path
    from collections import Counter
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    os.environ.setdefault("OPENCV_FFMPEG_LOGLEVEL", "-8")
    import cv2
    OUT_DIR = Path("/kaggle/working/check_le2i") if Path("/kaggle").exists() else Path("./check_le2i")
    if not (OUT_DIR / "state.pkl").exists():
        raise RuntimeError("Chưa có state.pkl (kernel chết trước mục 6). Hãy Run All lại từ đầu.")
    globals().update(pickle.load(open(OUT_DIR / "state.pkl", "rb")))
    print("Kernel đã restart: đã nạp lại trạng thái từ", OUT_DIR / "state.pkl")
cols = ["video_id", "group", "part", "no", "name", "label", "start", "end", "has_annot", "n_bbox",
        "fps", "width", "height", "n_frames_prop", "n_frames_dec", "codec", "excluded", "usable_event", "issues", "path", "ann_path"]
inv[cols].to_csv(OUT_DIR / "le2i_inventory.csv", index=False)

use = inv[~inv.excluded]
tot, nf = len(use), int((use.label == "fall").sum())
fps_set, size_set = set(use.fps.dropna()), set(zip(use.width.dropna().astype(int), use.height.dropna().astype(int)))
excl = inv[inv.excluded].groupby("group").agg(video=("path", "size"), co_annot=("has_annot", "sum"))
summary = dict(root=str(ROOT), videos_in_folder=len(inv), videos_used=tot, falls=nf, adl=int((use.label == "adl").sum()),
               no_annot_used=int((~use.has_annot).sum()), usable_event=int(use.usable_event.sum()),
               groups_used=sorted(set(use.group)), excluded=excl.to_dict("index"), fps=sorted(fps_set), sizes=sorted(size_set),
               bbox_layout=BBOX_LAYOUT, frame_base=FRAME_BASE, orphan_annotations=len(orphans),
               duplicate_video_ids=int(use.video_id.duplicated().sum()))
(OUT_DIR / "le2i_summary.json").write_text(json.dumps(summary, indent=2, default=str))

print("========== TỔNG KẾT LE2I ==========")
print(f"Toàn bộ thư mục: {len(inv)} video (bản gốc theo tài liệu ≈{EXPECTED['videos_full']})")
print("\nScene ĐÃ LOẠI: số video / số có annotation (kiểm chứng lý do loại):")
print(excl.to_string() if len(excl) else "  (không thấy Office/Lecture room trong thư mục)")
print(f"\nPHẠM VI DÙNG (Home + Coffee room): {tot} video | Fall: {nf} | ADL: {summary['adl']} | thiếu annotation: {summary['no_annot_used']}")
print(f"Scene trong phạm vi dùng: {summary['groups_used']}", "OK" if set(summary['groups_used']) == EXPECTED['groups'] else "⚠ KHÁC DỰ KIẾN (cần home + coffee_room)")
print(f"fps: {summary['fps']} (dự kiến {EXPECTED['fps']}) | kích thước: {summary['sizes']} (dự kiến {EXPECTED['size']})")
print(f"Dùng được cho đánh giá mức sự kiện: {summary['usable_event']}/{tot}")
print(f"Thứ tự cột bbox: {BBOX_LAYOUT} | frame annotation từ: {FRAME_BASE} | annotation mồ côi: {len(orphans)} | trùng video_id: {summary['duplicate_video_ids']}")
bad = inv[(inv.issues != "") & ~inv.excluded][["video_id", "name", "label", "issues"]]
print("\nVideo (trong phạm vi dùng) có cờ lỗi:" if len(bad) else "\nKhông có video nào trong phạm vi dùng bị cờ.")
print(bad.head(40).to_string() if len(bad) else "")
print("\nĐã ghi vào", OUT_DIR.resolve(), ":", sorted(p.name for p in OUT_DIR.iterdir()))

### Cách đọc kết quả
| Dấu hiệu | Ý nghĩa | Việc cần làm |
|---|---|---|
| Scene đã loại (Office, Lecture room) có `co_annot = 0` | Xác nhận lý do loại: annotation bị mất | Ghi vào DATA_CARD và báo cáo |
| Scene dùng ≠ {home, coffee_room} | Mirror thiếu Home/Coffee room, hoặc tên thư mục lạ | Xem lại cây thư mục, điền `DATA_ROOT` thủ công |
| `[THIEU_ANNOT]` | Video không có file annotation | Không dùng cho đánh giá mức sự kiện (`has_event_label=0`) |
| Annotation mồ côi | Có file `.txt` nhưng không có video cùng (scene, số) | Kiểm tra tên file / mirror thiếu video |
| `[TRUNG_VIDEO_ID]` | Hai video trùng scene+số | Kiểm tra thư mục lồng nhau/bản sao; sửa trước khi chuẩn hóa |
| `[KHOANG_NGA_LA]` | `start/end` không theo quy ước (một số bằng 0, hoặc end < start) | Xem video đó; không tự sửa nhãn |
| `[END_VUOT_SO_FRAME]` | Khoảng ngã vượt số frame video | Có thể annotation lệch với video: cần xem ảnh |
| Bbox lệch người trong ảnh ghép | Sai thứ tự cột hoặc sai gốc đánh số frame | Đổi `BBOX_LAYOUT` / `FRAME_BASE` rồi chạy lại |
| Số frame báo ≠ số frame giải mã | Metadata của `.avi` không đáng tin | Luôn dùng số frame giải mã thật khi chuẩn hóa |